# **1.)** OVERVIEW/CONTEXT

---
Text goes here....

---

<figure style="text-align:center;">
    <img src="notebook_images/umn_banner.png" alt="University of Minnesota">
</figure>

---
# <center>**<u>CERT x465-003</u>**: Data Interpretation and Application</center><br><center>Coursework and Assignments</center><br><center>Project Title: ***Deep Poking into Pokémon Data***</center>
<figure style="text-align:center;">
    <img src="notebook_images/pokes_pic.jpg" alt="Pokemon" width="1800">
</figure>

---

# **2.)** *<u>Software/Libraries Herein Implemented</u>*

---
More text goes here....

---

# **3.)** *<u>Assignments</u>*

---

## **3.1)** <u>Assignment 01</u>
- Completed by: Logan R. Sloan<br>

---
### Assignment 01 Details
- Write a short essay that address the following questions:
    - What problem or question are you trying to address?
    - Why are you interested in addressing this?
    - Why is this an important question or problem to address (in general)?
    - Who might your audience(s) be for this investigation?
    - How will the results of this investigation help your audience make decisions or draw conclusions?<br>

---
### Assignment 01 Essay


## **3.2)** <u>Assignment 02-A</u>
- Completed by: Logan R. Sloan<br>

---
### Assignment 02-A Details
- Identify at least one data source that can help answer the question you developed in Module 1.
1. First, do some research on your own to try and find a data source(s).
    - You might start with a simple Google search, you may ask someone  who has knowledge in the area, or you might already know of some candidates.
    - For example, if your question pertains to K-12 education in Minnesota, you might check out the Department of Education's data sources.
 2. After you have finished your search and identified good data source(s), please write a short essay that addresses to the following questions:
    - Describe the data source(s) that you found.
    - How is the data relevant to your question?
    - Where does the data come from?
    - How reliable is the source? Are there any signs of bias?<br>

---
### Assignment 02-A Essay

# **<u>NOTE</u>**: THE DATA SCRAPING SECTION SHOULD BE MOVED TO ASSIGNMENT 02A, SINCE THE POINT OF THE ASSIGNMENT IS TO FIND A DATASET TO USE AND THAT IS WHAT THE SCRAPING, CLEANING, ET CETERA IS DOING....

## **3.3)** <u>Assignment 03</u>
- Completed by: **Logan R. Sloan**<br>

---
### Assignment 03 Details
- Using the data source(s) that you identified, produce a 5-minute video describing what you are seeing in the data and what conclusions the data is pointing towards.
- Make sure you tie it back to the original question you set out to address.
- In the video, please also include a brief discussion of how groups with different perspectives might interpret the data differently.
- Post this video to share with your peers, and provide feedback on the videos of at least two peers.<br>

---

### **3.3.i)** Data Exploration for Assignment 03

---
#### Overview of Methodology to be Used to Completed Assignment 03
- Since I have chosen to create my own dataset, in order to complete *Assignment 03* the dataset must first be compiled together in a clean, usable format.
    - To do this I will, first, scrape multiple websites (and multiple pages on those site) for the base of my dataset.
    - Then I will cleanup this data where necessary, as scraping always involves data that isn't particularly useful in its "raw" state.
    - Lastly, I will expand upon and abstract from the cleaned data to create more columns/categories within the dataset to better aid my analysis of the dataset.
- To explore the dataset, I will use the paradigm of an Exploratory Data Analysis (EDA) and some other Data Science techniques to dig down into the data and see what interesting points in the pokemon dataset we can poke out and prop up for others' viewing.
    - This will include some comparison based observations, basic and more advanced statics calculations, visualizations, as well as some machine learning (clustering) analysis.

---

### **3.3.ii)** Scraping, Cleaning, and Expanding upon the Data to Build Dataset for Analysis

---

### **3.3.ii.0)** Python GPU Acceleration Activation

---

In [ ]:
%load_ext cudf.pandas
%load_ext cuml.accel

 ### **3.3.ii.1)** Python Library Imports for Scraping, Cleaning, and Expanding

---

In [ ]:
import os
import io
import re
import time
import urllib.request
import urllib.error
import pandas as pd
from bs4 import BeautifulSoup

BASE_URL = 'https://bulbapedia.bulbagarden.net'
DATA_DIR = os.path.expanduser('~/000_Duckspace/WanderduckDevelopment/Ducks/UMN/CERT-x465-003/data/')
_HEADERS = {'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64; rv:132.0) Gecko/20100101 Firefox/132.0'}

 ### **3.3.ii.2)** Base URL to Scrape

In [ ]:
 pokemon_ALL_url = "https://bulbapedia.bulbagarden.net/wiki/List_of_Pok%C3%A9mon_by_base_stats_in_Generation_IX"

### **3.3.ii.3)** Initial Scraper (Part 1)
- For part 1 of the initial scraper, most of the information from the tables will be pulled from the base url page (see image below for example).
<br>

<figure style="text-align:center;">
    <img src="notebook_images/initial_scrape_img00.png" alt="Bulbapedia Initial Scrape">
    <figcaption>
        The scraper looks through the HTML (right of image) that laysout the webpage.<br>
        The blue highlight shows that this particular part of the table contains both the Pokémon name and the link to it's individual page (to be used by the next scraper).
    </figcaption>
</figure>

In [ ]:
def scrape_pokemon_base_stats(url):
    """
    Scrapes the Bulbapedia base stats table using direct HTTP (no headless browser needed
    since the table is server-rendered HTML).
    Table columns: #, Pokémon (sprite), Pokémon.1 (name), HP, Attack, Defense,
                   Sp. Atk, Sp. Def, Speed, Total, Average
    Returns a list of dicts matching the first 10 columns of pokemon_dataset_MASTER.csv.
    """
    req = urllib.request.Request(url, headers={
        'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64; rv:132.0) Gecko/20100101 Firefox/132.0'
    })
    with urllib.request.urlopen(req, timeout=30) as resp:
        html = resp.read().decode('utf-8')

    dfs = pd.read_html(io.StringIO(html))
    df = next(
        t for t in dfs
        if 'HP' in t.columns and 'Attack' in t.columns
    )

    records = []
    for _, row in df.iterrows():
        try:
            pokedex_num = int(row['#'])
            name        = str(row['Pokémon.1']).strip()   # .1 suffix = name col; plain 'Pokémon' is the sprite (NaN)
            hp          = int(row['HP'])
            attack      = int(row['Attack'])
            defense     = int(row['Defense'])
            sp_atk      = int(row['Sp. Atk'])
            sp_def      = int(row['Sp. Def'])
            speed       = int(row['Speed'])
            total       = int(row['Total'])
            link = f"{BASE_URL}/wiki/{name.replace(' ', '_')}"
        except (ValueError, TypeError):
            continue

        records.append({
            'Pokedex Number': pokedex_num,
            'Pokemon':        name,
            'Link':           link,
            'HP':             hp,
            'Attack':         attack,
            'Defense':        defense,
            'Speed':          speed,
            'Special Attack': sp_atk,
            'Special Defense': sp_def,
            'Stat Total':     total,
        })

    return records


records = scrape_pokemon_base_stats(pokemon_ALL_url)
pokes_scraped = pd.DataFrame(records)
print(f"Scraped {len(pokes_scraped)} rows.")
pokes_scraped.head(10)

### **3.3.ii.3.x)** Save Initial Dataset

In [ ]:
pokes_scraped.to_csv(DATA_DIR + 'pokemon_base_stats_scraped.csv', index=False)
print(f"Saved {len(pokes_scraped)} rows to {DATA_DIR}pokemon_base_stats_scraped.csv")

### **3.3.ii.3.xx)** Cleanup Pokémon Names Where "Variations" Exist
- If you look at the preview of the initial scraped dataset, you'll see that some of the Pokémon have variations such as "Mega".
- It's a little off topic and overly technical to show how I identified all these (although this section here could probably be defined the same way).
- But the first code block below searches through the names and where variations exist it assigns the variation name to a new column in the dataset named "Variation".
- The second code block then removes the variation excess from the Pokémon's name, so that the standard Pokémon and the variants have the same name (think how you would categorize black dog and yellow dog).

In [ ]:
_VAR_PATTERN = re.compile(
    r'Mega.*[A-Za-z]|Alolan.*[A-Za-z]|Partner.*[A-Za-z]|Galarian.*[A-Za-z]'
    r'|Hisuian.*[A-Za-z]|Paldean.*[A-Za-z]+\D|in L.*[A-Za-z]|Primal.*[A-Za-z]'
    r'|[A-Za-z]+.Forme|[A-Za-z]+.Cloak|[A-Za-z]+.Rotom|[A-Za-z]+.Mode'
    r'|[A-Za-z]+.Kyurem|Ash.*[A-Za-z]|Eternal.*[A-Za-z]|[A-Za-z]+.Size'
    r'|[A-Za-z0-9]+%.Forme|Hoopa(?![\s\S]*?Hoopa).*[A-Za-z]'
    r'|D.*[A-Za-z]+.*Necrozma|[A-Za-z]+.Necrozma|Hero.*[A-Za-z]'
    r'|Crowned.*[A-Za-z]|Eternamax.*[A-Za-z]|\b\w+\s+Rider.*[A-Za-z]'
    r'|Bloodmoon.*[A-Za-z]|Male|Female'
)

variations = []
for name in pokes_scraped['Pokemon']:
    match = _VAR_PATTERN.search(name)
    variations.append(match.group() if match else '')

pokes_scraped.insert(10, 'Variation', variations)

n_var = sum(1 for v in variations if v != '')
print(f"Variation column added. {n_var} variants found out of {len(pokes_scraped)} rows.")
pokes_scraped[pokes_scraped['Variation'] != ''].head(10)

In [ ]:
pattern = r'Mega.*[A-Za-z]|Alolan.*[A-Za-z]|Partner.*[A-Za-z]|Galarian.*[A-Za-z]|Hisuian.*[A-Za-z]|Paldean.*[A-Za-z]+\D|in L.*[A-Za-z]|Primal.*[A-Za-z]|[A-Za-z]+.Forme|[A-Za-z]+.Cloak|[A-Za-z]+.Rotom|[A-Za-z]+.Mode|[A-Za-z]+.Kyurem|Ash.*[A-Za-z]|Eternal.*[A-Za-z]|[A-Za-z]+.Size|[A-Za-z0-9]+%.Forme|Hoopa(?![\s\S]*?Hoopa).*[A-Za-z]|D.*[A-Za-z]+.*Necrozma|[A-Za-z]+.Necrozma|Hero.*[A-Za-z]|Crowned.*[A-Za-z]|Eternamax.*[A-Za-z]|\b\w+\s+Rider.*[A-Za-z]|Bloodmoon.*[A-Za-z]|Male|Female'

# Removes the matching parts and clean up whitespace
pokes_scraped['Pokemon'] = pokes_scraped['Pokemon'].str.replace(pattern, '', regex=True).str.strip()

<figure style="text-align:center;">
    <img src="notebook_images/great-success.gif" alt="Great Success! Very Nice!">
    <figcaption>
        Very Nice! Part 1 of the initial scraping is completed!<br>
        Here's a trophy: <span style="display: inline-block; transform: rotate(180deg);">&#x1F3C6;</span>
    </figcaption>
</figure>

---

### **3.3.ii.4)** Initial Scraper (Part 2)
- For part 2 of the initial scraper, the links gathered for each individual Pokémon will all have to be gone through to pull more basic information to expand the dataset.
- Where part one was scraping one webpage, in part 2, one thousand two-hundred thirty-nine (1239 &#x1F480;) webpages will need to be scraped.
- The information to be gathered is Category, Type 1, Type 2, Height (m), and Weight (kg).
- Below is an example from one of these webpages.<br><br>

<figure style="text-align:center;">
    <img src="notebook_images/initial_scrape_img01.png" alt="Second of the Scrapes, Initially....">
    <figcaption>
        <span style="display: inline-block; transform: scale(-1, 1);">&#x1F446;</span>...."Seed Pokémon" under the name is the Category for reference....&#x1F446;
    </figcaption>
</figure>

### **NUMBERS)** Create Column Denoting Pokémon Generation
- There are nine generations of Pokémon apparently, and it might be worthwhile to examine changes over the generations
- The "Pokedex Number" column is the best way to do this because variants have the same number as their standard siblings (as opposed to index values or something else).
- The first line of code creates an empty column named "Generation", the all the following lines fill-in which generation number.

In [ ]:
pokes_scraped['Generation'] = np.nan
pokes_scraped.loc[pokes_scraped['Pokedex Number'] <= 151, 'Generation'] = 1
pokes_scraped.loc[pokes_scraped['Pokedex Number'] > 151, 'Generation'] = 2
pokes_scraped.loc[pokes_scraped['Pokedex Number'] > 251, 'Generation'] = 3
pokes_scraped.loc[pokes_scraped['Pokedex Number'] > 386, 'Generation'] = 4
pokes_scraped.loc[pokes_scraped['Pokedex Number'] > 493, 'Generation'] = 5
pokes_scraped.loc[pokes_scraped['Pokedex Number'] > 649, 'Generation'] = 6
pokes_scraped.loc[pokes_scraped['Pokedex Number'] > 721, 'Generation'] = 7
pokes_scraped.loc[pokes_scraped['Pokedex Number'] > 809, 'Generation'] = 8
pokes_scraped.loc[pokes_scraped['Pokedex Number'] > 905, 'Generation'] = 9

---

### **NUMBERS)** Pokémon Evolution Stage Scraper
- Pokémon evolve as they get stronger or something: People train them and they grow into more ridiculous versions of themselves and this is termed "evolution" in the lore, I think.
- Tracking how the Pokémon change as they evolve will add great insight into the analysis.
- Pokémon, as far as I can tell, either have a two-stage evolution (first stage changes into second/final) or have a three-stage evolution (where the first stage changes into second/intermediate and second into third/final).
- Also, it should be noted that not all Pokémon evolve; as you will see in the analysis, only about 1/3 of Pokémon evolve into other Pokémon.

### **NUMBERS)** Evolution URLs
- I found separate pages on the same website that organize the two-stage evolutions from the three-stage
- This makes the scraping much easier, but it does require rolling through two webpages instead of a single page, but this written to be extremely fast, so no problem....

In [ ]:
evo3_url = 'https://pokemon.fandom.com/wiki/List_of_Pok%C3%A9mon_by_three-stage_evolution'
evo2_url = 'https://pokemon.fandom.com/wiki/List_of_Pok%C3%A9mon_by_two-stage_evolution'

### **NUMBERS)** Evolution Stage Scraper

In [ ]:
DATA_DIR = os.path.expanduser('~/000_Duckspace/WanderduckDevelopment/Ducks/UMN/CERT-x465-003/data/')


async def get_pokes_rows_pw(page, url):
    """Navigate to url and return all non-header rows from .prettytable tables."""
    await page.goto(url, wait_until='domcontentloaded')
    await page.wait_for_selector('.mw-parser-output .prettytable')
    rows = []
    for table in await page.locator('.mw-parser-output .prettytable').all():
        for i, row in enumerate(await table.locator('tr').all()):
            if i > 0:
                rows.append(row)
    return rows


async def get_name(cell):
    """Return the Pokémon name from a table cell (second anchor if present, else first, else cell text)."""
    anchors = await cell.locator('a').all()
    if len(anchors) >= 2:
        return await anchors[1].inner_text()
    elif len(anchors) == 1:
        return await anchors[0].inner_text()
    return await cell.inner_text()


async def pokes_parser1_pw(row):
    """Parse a three-stage evolution table row → dict with ev1/ev2/ev3 keys."""
    cells = await row.locator('td').all()
    n = len(cells)
    if n == 3:
        return {'ev1': await get_name(cells[0]), 'ev2': await get_name(cells[1]), 'ev3': await get_name(cells[2])}
    elif n == 2:
        return {'ev2': await get_name(cells[0]), 'ev3': await get_name(cells[1])}
    else:
        return {'ev3': await get_name(cells[0])}


async def pokes_parser2_pw(row):
    """Parse a two-stage evolution table row → dict with ev1/ev2 keys."""
    cells = await row.locator('td').all()
    n = len(cells)
    if n == 2:
        first_anchors = await cells[0].locator('a').all()
        ev1 = await first_anchors[1].inner_text() if len(first_anchors) >= 2 else await cells[0].inner_text()
        ev2 = await get_name(cells[1])
        return {'ev1': ev1, 'ev2': ev2}
    else:
        cell_anchors = await cells[0].locator('a').all()
        ev2 = await cell_anchors[1].inner_text() if len(cell_anchors) >= 2 else await cells[0].inner_text()
        return {'ev2': ev2}


async def main():
    async with async_playwright() as pw:
        browser = await pw.firefox.launch(headless=True)
        page = await browser.new_page()

        rows3 = await get_pokes_rows_pw(page, evo3_url)
        evo3_data = [await pokes_parser1_pw(r) for r in rows3]
        pd.DataFrame(evo3_data).to_csv(DATA_DIR + 'evo3.csv', index=False)

        rows2 = await get_pokes_rows_pw(page, evo2_url)
        evo2_data = [await pokes_parser2_pw(r) for r in rows2]
        pd.DataFrame(evo2_data).to_csv(DATA_DIR + 'evo2.csv', index=False)

        await browser.close()

await main()
print("Scraping complete. CSVs saved to data directory.")

### **NUMBERS)** Cleaning-up the Evolution Data {***<u>WARNING</u>***: Long-winded explanation of what's happening below--PLEASE SKIP!!!}
- The scraper above creates two seperate `.csv` files (one for each url).
- So, the two will need to be merged together and then merged into the larger, main dataset, but before that can be done, the evolution stage data needs to be turned into integers.
- The reason why the data wasn't scraped as integers is because Python doesn't allow naming variables with only numbers.
- Python variables must start with either an uppercase or lowercase letter [a-z, A-Z] or an underscore [ _ ].
- And when pulling information from a website like this, the data must be stored as a variable to be compiled together.
- First, the data from each website is compiled into what's called a dictionary.
    - Dictionaries work sort of like cabinets:
        1. There are *unique* "keys" that function like each cabinet box in your kitchen [you cannot have keys of the same value just like every cabinet is a totally separate space], and
        2. There are *non-unique* "values" within those keys that can repeat (like having two of the same pots and whatnot).
- Second, the two separate dictionaries are saved as `.csv` files.
    - A `.csv` is used for three reasons:
        1.  In case something messes up in the integer conversion or merging, we won't need to run the scraper again (and can simply work on fixing our crap code).
        2. because `.csv` files can easily loaded into what are called DataFrames, and these are very easy to manipulate and merge together.
        3. There are certain Pokémon that have the ability to evolve into more than one, singular Pokémon in the next stage, and the way the website's table is setup, I could not find a good work around to match these separate evolutions.
            - [Reference the parenthetical from #1]--This being the case and my code being crap, a `.csv` file is human-readable and easy to manually edit.
            - If you *must* know, it was easy to edit because the Pokémon missing their evolution stages are directly below the stages they are missing, so I just ran a script that found blanks, moved a row up, copied the correct number of values, and pasted them in the blanks.
            - See the picture below to better understand how the table is set up: because of how HTML functions, the Pokémon "Bellossom" is technically on a row by itself, where as "Vileplume" is rowed with the first two.
<figure style="text-align:center;">
    <img src="notebook_images/evo_scrape_img00.png" alt="Gloom of HTML">
    <figcaption>"Gloom" is my natural habitat....
        <span style="display: inline-block; transform: scale(1.5, -1);">&#x1F984;</span>
    </figcaption>
</figure>
<br>

---
- This all means that cleaning the code will work in this order:
    1. Manually filling in the missing Pokémon for the edge cases.
    2. Loading in the separate `.csv` files.
    3. Merging, one-at-a-time, the two `.csv` files into the main dataset
    4. Creating two new columns in the main dataset from the non-integer data (which works to convert it to integers in the process).
        1. <u>"Evolution Stage"</u>: This is the integer value of the Pokémon's place in the evolution chain (1, 2, or 3--and 0 for Pokémon that don't evolve).
        2. <u>"Evolves"</u>: This is a boolean value (i.e. True or False) that tells us whether a Pokémon evolves or not. This is to make it easier to analyze non-evolving Pokémon vs. evolving Pokémon.

In [ ]:
evo3 = pd.read_csv(DATA_DIR + 'evo3.csv')
evo2 = pd.read_csv(DATA_DIR + 'evo2.csv')


# Merge three-stage evolution data into df
df['ev1'] = df['Pokemon'].isin(evo3['ev1'])
df['ev2'] = df['Pokemon'].isin(evo3['ev2'])
df['ev3'] = df['Pokemon'].isin(evo3['ev3'])

df.loc[df['ev1'] == True, ['Evolution Stage', 'Evolves']] = [1, True]
df.loc[df['ev2'] == True, ['Evolution Stage', 'Evolves']] = [2, True]
df.loc[df['ev3'] == True, ['Evolution Stage', 'Evolves']] = [3, False]

df.drop(['ev1', 'ev2', 'ev3'], axis=1, inplace=True)

# Merge two-stage evolution data into df
df['ev1'] = df['Pokemon'].isin(evo2['ev1'])
df['ev2'] = df['Pokemon'].isin(evo2['ev2'])

df.loc[df['ev1'] == True, ['Evolution Stage', 'Evolves']] = [1, True]
df.loc[df['ev2'] == True, ['Evolution Stage', 'Evolves']] = [2, False]

df.drop(['ev1', 'ev2'], axis=1, inplace=True)

# Fill remaining NaN evolution stages (non-evolving Pokémon)
df.loc[df['Evolution Stage'].isna(), ['Evolution Stage', 'Evolves']] = [0, False]

print("Cleaning and merging complete.")

### **NUMBERS)** Probably a Good Time to Save Our Progress

In [ ]:
pokes_scraped.to_csv(('~/000_Duckspace/WanderduckDevelopment/Ducks/UMN/CERT-x465-003/data/pokemon_dataset_MASTER.csv', index=False))

###

### **3.3.iii)** Exploratory Data Analysis (EDA)

---

#### EDA Overview
- This section contains the full Exploratory Data Analysis for Assignment 03.
- The dataset used is `pokemon_dataset_MASTER.csv` (1,231 rows × 55 columns), custom-built from scraped Bulbapedia and Pokémon Fandom Wiki data.
- Eight thematic questions are explored, followed by an unsupervised machine learning clustering capstone.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

# Consistent style
sns.set_theme(style='darkgrid', palette='viridis')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (12, 6)

STAT_COLS   = ['HP', 'Attack', 'Defense', 'Speed', 'Special Attack', 'Special Defense']
ROLE_ORDER  = ['Physical Sweeper', 'Special Sweeper', 'Mixed Attacker',
               'Balanced', 'Tank', 'Physical Wall', 'Special Wall']
PCT_COLS    = ['HP_pct', 'Attack_pct', 'Defense_pct',
               'Speed_pct', 'Special_Attack_pct', 'Special_Defense_pct']
ZSCORE_COLS = ['HP_zscore', 'Attack_zscore', 'Defense_zscore',
               'Speed_zscore', 'Special_Attack_zscore', 'Special_Defense_zscore']

DATA_DIR = os.path.expanduser(
    '~/000_Duckspace/WanderduckDevelopment/Ducks/UMN/CERT-x465-003/data/'
)
pokes = pd.read_csv(DATA_DIR + 'pokemon_dataset_MASTER.csv')

In [ ]:
# --- Dataset Verification ---
assert pokes.shape == (1231, 55), f"Expected (1231, 55), got {pokes.shape}"
assert 'BMI' in pokes.columns,             "Missing column: BMI"
assert 'Role' in pokes.columns,            "Missing column: Role"
assert 'Stat Std Dev' in pokes.columns,    "Missing column: Stat Std Dev"
assert 'Legendary' in pokes.columns,       "Missing column: Legendary"
assert 'Evolution Stage' in pokes.columns, "Missing column: Evolution Stage"
for col in ZSCORE_COLS:
    assert col in pokes.columns, f"Missing z-score column: {col}"

print(f"Dataset loaded: {pokes.shape[0]} rows × {pokes.shape[1]} columns")
print(f"\nRole distribution:\n{pokes['Role'].value_counts().to_string()}")
print(f"\nMissing values in key columns:\n"
      f"{pokes[['BMI','Role','Stat Std Dev','Legendary','Evolution Stage']].isna().sum().to_string()}")

### **3.3.iii.1)** Section 1: Body Composition and Combat Role

---
**Question:** Does a Pokémon's physical body shape (BMI, height, weight) predict how it fights?

In [ ]:
# --- 1a: BMI Distribution ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(pokes['BMI'].clip(upper=200), bins=60, kde=True,
             ax=axes[0], color='steelblue')
axes[0].set_title('BMI Distribution (clipped at 200)')
axes[0].set_xlabel('BMI')

bmi_filtered = pokes[pokes['BMI'] < 200]
sns.histplot(bmi_filtered['BMI'], bins=60, kde=True,
             ax=axes[1], color='coral')
axes[1].set_title('BMI Distribution (BMI < 200)')
axes[1].set_xlabel('BMI')

plt.suptitle("Pokémon BMI Distribution", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nTop 10 highest-BMI Pokémon:")
print(pokes.nlargest(10, 'BMI')[
    ['Pokemon', 'Type 1', 'Height (m)', 'Weight (kg)', 'BMI', 'Role']
].to_string(index=False))

In [ ]:
# --- 1b: BMI by Combat Role ---
fig, ax = plt.subplots(figsize=(12, 6))
bmi_data = pokes[pokes['BMI'] < 200].copy()

sns.violinplot(
    data=bmi_data,
    x='Role', y='BMI',
    order=ROLE_ORDER,
    palette='Set2',
    inner='box',
    ax=ax
)
ax.set_title('BMI Distribution by Combat Role', fontsize=14, fontweight='bold')
ax.set_xlabel('Combat Role')
ax.set_ylabel('BMI')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# --- 1c: Height vs Weight scatter (log scale, interactive) ---
fig = px.scatter(
    pokes[pokes['BMI'] < 500],
    x='Height (m)', y='Weight (kg)',
    color='Type 1',
    hover_name='Pokemon',
    hover_data=['BMI', 'Role', 'Generation'],
    log_x=True, log_y=True,
    title='Height vs. Weight by Primary Type (log-log scale)',
    template='plotly_dark',
    opacity=0.7
)
fig.show()

In [ ]:
# --- 1d: Median BMI by Generation ---
gen_bmi = pokes.groupby('Generation')['BMI'].median().reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=gen_bmi, x='Generation', y='BMI', palette='viridis', ax=ax)
ax.set_title('Median Pokémon BMI by Generation', fontsize=14, fontweight='bold')
ax.set_xlabel('Generation')
ax.set_ylabel('Median BMI')
plt.tight_layout()
plt.show()

print("\nMedian BMI per Generation:")
print(gen_bmi.to_string(index=False))

### **3.3.iii.2)** Section 2: Type Shapes Stat Identity, Not Just Stat Total

---
**Question:** Does a Pokémon's primary type determine its fighting personality more than its raw power level?

In [ ]:
# --- 2a: Combat ratio profiles by Type 1 ---
type_ps = pokes.groupby('Type 1')['Physical/Special'].median().sort_values()
type_od = pokes.groupby('Type 1')['Offensive/Defensive'].median().sort_values()

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

type_ps.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].axvline(1.0, color='red', linestyle='--', linewidth=1.5, label='Balanced (1.0)')
axes[0].set_title('Median Physical/Special Ratio by Type', fontweight='bold')
axes[0].set_xlabel('Physical / Special')
axes[0].legend()

type_od.plot(kind='barh', ax=axes[1], color='coral')
axes[1].axvline(1.0, color='red', linestyle='--', linewidth=1.5, label='Balanced (1.0)')
axes[1].set_title('Median Offensive/Defensive Ratio by Type', fontweight='bold')
axes[1].set_xlabel('Offensive / Defensive')
axes[1].legend()

plt.suptitle('Combat Personality by Primary Type', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- 2b: Stat fingerprint radar charts (6 focal types) ---
FOCUS_TYPES = ['Ghost', 'Psychic', 'Fighting', 'Rock', 'Steel', 'Dragon']
labels = ['HP', 'Attack', 'Defense', 'Speed', 'Sp. Atk', 'Sp. Def']
N = len(labels)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, axes = plt.subplots(2, 3, figsize=(15, 10), subplot_kw=dict(polar=True))
axes = axes.flatten()

for i, ptype in enumerate(FOCUS_TYPES):
    means = pokes[pokes['Type 1'] == ptype][PCT_COLS].mean().values.tolist()
    means += means[:1]
    axes[i].plot(angles, means, 'o-', linewidth=2)
    axes[i].fill(angles, means, alpha=0.25)
    axes[i].set_xticks(angles[:-1])
    axes[i].set_xticklabels(labels)
    axes[i].set_title(f'{ptype} Type', fontweight='bold', pad=15)
    axes[i].set_ylim(0, 0.25)

plt.suptitle('Stat Profile "Fingerprint" by Type (% of Stat Total)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- 2c: Does dual typing affect stat profile shape? ---
pokes['Type Label'] = pokes['Dual Type'].map({True: 'Dual Type', False: 'Mono Type'})

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.violinplot(data=pokes, x='Type Label', y='Physical/Special',
               palette='Set1', inner='box', ax=axes[0])
axes[0].axhline(1.0, color='black', linestyle='--', linewidth=1.2)
axes[0].set_title('Physical/Special by Type Count', fontweight='bold')

sns.violinplot(data=pokes, x='Type Label', y='Offensive/Defensive',
               palette='Set1', inner='box', ax=axes[1])
axes[1].axhline(1.0, color='black', linestyle='--', linewidth=1.2)
axes[1].set_title('Offensive/Defensive by Type Count', fontweight='bold')

plt.suptitle('Does Dual Typing Affect Stat Identity?', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nMedian ratios — Mono vs. Dual Type:")
print(pokes.groupby('Type Label')[['Physical/Special', 'Offensive/Defensive']].median().round(3).to_string())